# Methodology, assumptions and data quality

A full description for the reviewer: how the forecast is built, which assumptions are baked in, and where the data is "dirty". (Detailed walkthrough up front; the calculations follow below.)

---

## 0. Scope

- **What we forecast:** `consumption_usd` (accrued consumption revenue) and `cash` (actual cash receipts) for **June–August 2026**.
- **Three scenarios:** `downside` / `base` / `upside`.
- **Forecast date (as-of):** `2026-06-06` — the last invoice date in the data. Everything before it is actual, after it is forecast. June is treated as a forecast month (only 6 days of June are in the data).
- **Units:** final tables are in thousands of USD (`_k`).

---

## 1. Consumption forecast (`df_historical_usage`)

Computed at the **customer × product × month** grain, then aggregated.

**1.1. Customer base.** Only `customer_status = 'active'` (churned / paused are excluded up front — they won't be in the forecast period).

**1.2. Volume anchor.** Take the actual **May 2026** volume per customer×product and carry it forward with a growth coefficient.

**1.3. Growth coefficients (MoM)** — from the March→April→May 2026 dynamics:

| Scenario | Formula | Clamp (bounds) |
|---|---|---|
| downside (`least`) | min of the two MoM ratios | from −10% to 0% → `[0.90, 1.00]` |
| base (`avg`) | geometric mean of the two MoM ratios | from −5% to +5% → `[0.95, 1.05]` |
| upside (`greatest`) | max of the two MoM ratios | from 0% to +10% → `[1.00, 1.10]` |

- **Large partners `CUST-0006`, `CUST-0018`** — individual coefficient (per customer×product), since they heavily drive the total.
- **Everyone else** — per-product coefficient (overall product trend).
- **Compounding:** June = coef¹, July = coef², August = coef³.

**1.4. Price.** Base price list (from 2024-01; `gpu_accelerated` from 2025-10) with price changes: `compute_standard` from 2025-07, `object_storage` from 2026-01. Resolved via `coalesce(new_object_storage, new_compute, base)`.

**1.5. Discounts.** From `contracts` (except `CON-1888`). If the contract is `active` + `auto_renewal` → treated as valid through the end of the window; otherwise — through the actual `contract_end`, **after which the discount drops to 0**.

**1.6. Misc.** `gpu_a100` + `gpu_accelerated` merged into a single `gpu`. `revenue = quantity × (1 − discount) × price`.

**1.7. New CRM deals** (`df_new_customers`) — added as monthly consumption:

| Deal | downside | base | upside |
|---|:---:|:---:|:---:|
| OPP-1039 (Closed Won) | ✅ | ✅ | ✅ |
| OPP-1036 (Commit) | — | ✅ | — |
| OPP-1037 (Commit) | — | — | ✅ |

`CUST-0038`/`CUST-0047` are zeroed out **only in the downside** (unclear CRM status). `CUST-0016`, `CUST-0014`, `OPP-1035` — not included.

---

## 2. Cash forecast (`df_cash_scenarios`)

Cash ≠ consumption: money arrives with a lag and depends on the contract type. Three separate components (non-overlapping → no double counting):

> **Cash = A. Existing receivables + B. Forecast consumption × lag + C. New deals**

- **A. Receivables** — real invoices issued and unpaid as of as-of. Near-term ones (`due+7` inside the window) are collected in full in their month; overdue ones — with recovery by debt age, spread over 3 months. Identical across all scenarios.
- **B. Forecast × lag** — apply the historical payment lag (≈16% after 1 mo, ≈56% after 2 mo) to June–August consumption (arrears customers only). The past is not taken here — it is already in A.
- **C. Deals** — `annual_prepay` deals pay a year upfront in the start month: `committed × 12 / term`.

More detail in the "Cash forecast logic" markdown block below.

---

## 3. Assumptions

| # | Assumption | Where |
|---|---|---|
| 1 | Growth extrapolated from 3 months (Mar–May) with ±10%/±5% clamps — outlier protection | Consumption 1.3 |
| 2 | May is a representative volume anchor | Consumption 1.2 |
| 3 | Discount applies only during the contract term; without `auto_renewal` → 0 after it ends | Consumption 1.5 |
| 4 | OPP-1039 / OPP-1036 / OPP-1037 are all `annual_prepay`, paid a year upfront in one payment | Cash C |
| 5 | Existing `annual_prepay` customers generate no new cash in the window (prepayment happened earlier) | Cash B |
| 6 | `quarterly` customers are put in the lag as ordinary arrears (quarter phase unknown) | Cash B |
| 7 | Receivables recovery rates 80 / 50 / 20 / 5% — expert judgment, not from collection facts | Cash A |
| 8 | Overdue debt is collected evenly over 3 months (`/3`) | Cash A |
| 9 | Payment lag is global (whole history), not per individual customer | Cash B |
| 10 | Receivables do not depend on the scenario (it is already-existing debt) | Cash A |

---

## 4. Where the data lacks cleanliness (data-quality flags)

| # | Issue | Workaround / risk |
|---|---|---|
| 1 | **CON-1888** (contract behind OPP-1039) — `signed_not_started`, account unmatched | Excluded from contracts, the deal is handled via CRM. The OPP-1039 ↔ CON-1888 link is an assumption, risk of double counting |
| 2 | **Vertex Labs OPP-1035/1036/1037** — three deals on one account with the same date | We take only one (OPP-1036 base / OPP-1037 upside), not together; OPP-1035 is not used. Classic double-counting signal |
| 3 | **CUST-0038 / CUST-0047** — ambiguous CRM status (OPP-1017/OPP-1021) | Zeroed only in the downside; in base/upside — treated as organic |
| 4 | **March 2026** — GPU capacity constraint distorts MoM | Coefficient clamps smooth the outlier |
| 5 | **Billing-system change (April 2026)** | April invoices are questionable → affects receivables accuracy |
| 6 | **Concentration** — `network_egress` hinges on 1–2 large customers | Large partners (CUST-0006/0018) carved out into a separate coefficient |
| 7 | **Receivables >180 days ≈ 999K** | Largely bad debt, recovery 5% — but the exact rate must be validated with AR |
| 8 | **Duplicates and test invoices** | Dedup by `invoice_id`, `test_invoice` and `void_pending` excluded |
| 9 | **No actual collection outcomes** in the data | Recovery rates cannot be calibrated — pure assumption |

---

## 5. Open questions

- **Sales:** the real structure of the Vertex deals (OPP-1035/1036/1037) — separate amounts or one umbrella record? This drives base/upside.
- **Finance / Legal:** the contract type of OPP-1039 (really `annual_prepay`?) and its link to CON-1888 — affects 1,800K of June cash.
- **CRM:** statuses of CUST-0038 / CUST-0047 / CUST-0016 / CUST-0014.
- **AR team:** recovery rates by receivables age and the order of overdue collection.

In [1]:
from db import get_con
import plotly.express as px
import pandas as pd

In [2]:
con = get_con()

Setting up database tables...
Done. All tables are ready.


# Forecast for current customers (usage)

In [3]:
df_historical_usage = con.sql(
    """
    with
        -- Collect all pricing information
        cte_basic_price as
        (select product,
                list_price_usd as basic_price_usd
         from price_list
         where 1=1
         -- Base-price selection conditions are written below
         and ((product = 'compute_standard' and date(effective_from) = date('2024-01-01'))
              or (product = 'gpu_a100' and date(effective_from) = date('2024-01-01'))
              or (product = 'object_storage' and date(effective_from) = date('2024-01-01'))
              or (product = 'network_egress' and date(effective_from) = date('2024-01-01'))
              or (product = 'gpu_accelerated' and date(effective_from) = date('2025-10-01'))))

        , cte_billing_customer as
        (select distinct
                customer_id,
                segment
         from billing_customers
         where customer_status = 'active')

       , cte_price_compute_standard_for_new as
        (select product,
                list_price_usd as price_usd,
                date(effective_from) as effective_from
         from price_list
         where product = 'compute_standard'
           and date(effective_from) = date('2025-07-01'))

       , cte_price_object_storage as
        (select product,
                list_price_usd as price_usd,
                date(effective_from) as effective_from
         from price_list
         where product = 'object_storage'
           and date(effective_from) = date('2026-01-01'))

       , cte_contracts_basic_info as
        (select customer_id,
                date_trunc('month', contract_start) as contract_start,
                case
                when contract_status = 'active'
                 and auto_renewal
                then date('2026-10-01') -- Set the contract end after the forecast period

                when contract_status != 'active'
                then date(date_trunc('month', contract_end)) -- Take the actual contract end date

                when not auto_renewal
                then date(date_trunc('month', contract_end)) -- Take the actual contract end date
                end as contract_end,

                discount_pct*0.01 as discount_percent
         from contracts
         where contract_id != 'CON-1888')

        , cte_usage_fact as
        (SELECT date(month) as pmonth,
                customer_id,
                product,

                sum(case product
                    when 'compute_standard' then compute_hours
                    when 'gpu_a100' then gpu_hours
                    when 'gpu_accelerated' then gpu_hours
                    when 'network_egress' then egress_tb
                    when 'object_storage' then storage_tb_avg
                    end) as quantity
         FROM monthly_usage
         where customer_id in (select customer_id from cte_billing_customer) -- Immediately exclude customers that will NOT be in the forecast period
         GROUP BY 1,2,3)

        , coef_by_users as
        (select customer_id,
                product,

                -- Negative: from -10% to 0% MoM
                greatest(0.90,
                         least(1.00,
                               if(q_03 = 0, 1.00, q_04 / q_03),
                               if(q_04 = 0, 1.00, q_05 / q_04))) as least_coef,

                -- Positive: from 0% to +10% MoM
                least(1.10,
                      greatest(1.00,
                               if(q_03 = 0, 1.00, q_04 / q_03),
                               if(q_04 = 0, 1.00, q_05 / q_04))) as greatest_coef,

                -- Base: from -5% to +5% MoM
                greatest(0.95,
                         least(1.05,
                               sqrt(if(q_03 = 0, 1.00, q_04 / q_03)
                                       * if(q_04 = 0, 1.00, q_05 / q_04)))) as avg_coef
         from (select customer_id,
                      product,
                      sum(if(pmonth = date('2026-03-01'), quantity, 0)) as q_03,
                      sum(if(pmonth = date('2026-04-01'), quantity, 0)) as q_04,
                      sum(if(pmonth = date('2026-05-01'), quantity, 0)) as q_05
               from cte_usage_fact
               where customer_id in ('CUST-0006', 'CUST-0018') -- Specific large partners
                 and pmonth >= date('2026-03-01')
                 and pmonth <= date('2026-05-01')
               group by 1,2) as rd)

        , coef_by_product as
        (select product,

                -- Negative: from -10% to 0% MoM
                greatest(0.90,
                         least(1.00,
                               if(q_03 = 0, 1.00, q_04 / q_03),
                               if(q_04 = 0, 1.00, q_05 / q_04))) as least_coef,

                -- Positive: from 0% to +10% MoM
                least(1.10,
                      greatest(1.00,
                               if(q_03 = 0, 1.00, q_04 / q_03),
                               if(q_04 = 0, 1.00, q_05 / q_04))) as greatest_coef,

                -- Base: from -5% to +5% MoM
                greatest(0.95,
                         least(1.05,
                               sqrt(if(q_03 = 0, 1.00, q_04 / q_03)
                                       * if(q_04 = 0, 1.00, q_05 / q_04)))) as avg_coef
         from (select product,
                      sum(if(pmonth = date('2026-03-01'), quantity, 0)) as q_03,
                      sum(if(pmonth = date('2026-04-01'), quantity, 0)) as q_04,
                      sum(if(pmonth = date('2026-05-01'), quantity, 0)) as q_05
               from cte_usage_fact us_f
               where customer_id not in ('CUST-0006', 'CUST-0018') -- Specific large partners
                 and pmonth >= date('2026-03-01')
                 and pmonth <= date('2026-05-01')
               group by 1) as rd)

        -- Forecast scaffold
        , cte_future_months AS
        (SELECT *
         FROM (VALUES
                (DATE '2026-06-01'),
                (DATE '2026-07-01'),
                (DATE '2026-08-01')) AS t(pmonth))

        , cte_customer_product AS
        (SELECT DISTINCT customer_id,
                         product,
                         quantity
         FROM cte_usage_fact
         where date(pmonth) = date('2026-05-01'))

        , cte_future_customer_product_and_month AS
        (SELECT fm.pmonth,
                cp.customer_id,
                cp.product,
                cp.quantity
         FROM cte_customer_product cp
         CROSS JOIN cte_future_months fm)

        , cte_future_customer_product as
        (select cfcpm.pmonth,
                cfcpm.customer_id,
                cfcpm.product,
                cfcpm.quantity,

                -- Minimum
                case
                when cfcpm.pmonth = date('2026-06-01')
                then cfcpm.quantity
                     * coalesce(c_u.least_coef, c_p.least_coef, 1.00)

                when cfcpm.pmonth = date('2026-07-01')
                then cfcpm.quantity
                     * coalesce(c_u.least_coef, c_p.least_coef, 1.00)
                     * coalesce(c_u.least_coef, c_p.least_coef, 1.00)

                when cfcpm.pmonth = date('2026-08-01')
                then cfcpm.quantity
                     * coalesce(c_u.least_coef, c_p.least_coef, 1.00)
                     * coalesce(c_u.least_coef, c_p.least_coef, 1.00)
                     * coalesce(c_u.least_coef, c_p.least_coef, 1.00)
                end as least_quantity,

                -- Maximum
                case
                when cfcpm.pmonth = date('2026-06-01')
                then cfcpm.quantity
                     * coalesce(c_u.greatest_coef, c_p.greatest_coef, 1.00)

                when cfcpm.pmonth = date('2026-07-01')
                then cfcpm.quantity
                     * coalesce(c_u.greatest_coef, c_p.greatest_coef, 1.00)
                     * coalesce(c_u.greatest_coef, c_p.greatest_coef, 1.00)

                when cfcpm.pmonth = date('2026-08-01')
                then cfcpm.quantity
                     * coalesce(c_u.greatest_coef, c_p.greatest_coef, 1.00)
                     * coalesce(c_u.greatest_coef, c_p.greatest_coef, 1.00)
                     * coalesce(c_u.greatest_coef, c_p.greatest_coef, 1.00)
                end as greatest_quantity,

                -- Average
                case
                when cfcpm.pmonth = date('2026-06-01')
                then cfcpm.quantity
                     * coalesce(c_u.avg_coef, c_p.avg_coef, 1.00)

                when cfcpm.pmonth = date('2026-07-01')
                then cfcpm.quantity
                     * coalesce(c_u.avg_coef, c_p.avg_coef, 1.00)
                     * coalesce(c_u.avg_coef, c_p.avg_coef, 1.00)

                when cfcpm.pmonth = date('2026-08-01')
                then cfcpm.quantity
                     * coalesce(c_u.avg_coef, c_p.avg_coef, 1.00)
                     * coalesce(c_u.avg_coef, c_p.avg_coef, 1.00)
                     * coalesce(c_u.avg_coef, c_p.avg_coef, 1.00)
                end as avg_quantity
         from cte_future_customer_product_and_month cfcpm
         left join coef_by_users c_u
                on cfcpm.customer_id = c_u.customer_id
                    and cfcpm.product = c_u.product
         left join coef_by_product c_p
                on cfcpm.product = c_p.product)

        -- Combine actuals and forecast
        , cte_usage as
        (select pmonth,
                customer_id,
                product,
                quantity,
                null as least_quantity,
                null as greatest_quantity
         from cte_usage_fact

         union all

         select pmonth,
                customer_id,
                product,
                avg_quantity as quantity,
                least_quantity,
                greatest_quantity
         from cte_future_customer_product)

    select usg.pmonth,
           usg.customer_id,
           case usg.product
           when 'compute_standard' then 'compute_standard'
           when 'gpu_a100' then 'gpu'
           when 'gpu_accelerated' then 'gpu'
           when 'network_egress' then 'network_egress'
           when 'object_storage' then 'object_storage'
           end as product,
           if(usg.customer_id in ('CUST-0038', 'CUST-0047'),
              0, least_quantity) as least_quantity,
           quantity,
           greatest_quantity,

           coalesce(cpos.price_usd, cpcs.price_usd, basic_price_usd) as price,
           coalesce(discount_percent, 0.00) as discount_percent,

           -- exclude 'CUST-0038', 'CUST-0047' because their CRM statuses are unclear
           if(usg.customer_id in ('CUST-0038', 'CUST-0047'),
              0,
              least_quantity
               * (1-coalesce(discount_percent, 0.00))
               * coalesce(cpos.price_usd, cpcs.price_usd, basic_price_usd)) as least_revenue,

           quantity
              * (1-coalesce(discount_percent, 0.00))
              * coalesce(cpos.price_usd, cpcs.price_usd, basic_price_usd) as revenue,

           greatest_quantity
               * (1-coalesce(discount_percent, 0.00))
               * coalesce(cpos.price_usd, cpcs.price_usd, basic_price_usd) as greatest_revenue
    from cte_usage usg
    left join cte_basic_price cbp
           on usg.product = cbp.product
    left join cte_price_compute_standard_for_new cpcs
           on usg.product = cpcs.product
              and date(usg.pmonth) >= date(cpcs.effective_from)
    left join cte_price_object_storage cpos
           on usg.product = cpos.product
              and date(usg.pmonth) >= date(cpos.effective_from)
    left join cte_contracts_basic_info ccbi
           on usg.customer_id = ccbi.customer_id
              and date(usg.pmonth) >= date(ccbi.contract_start)
              and date(usg.pmonth) <= date(ccbi.contract_end)
    """).df()

In [4]:
df_historical_usage

,pmonth,customer_id,product,least_quantity,quantity,greatest_quantity,price,discount_percent,least_revenue,revenue,greatest_revenue
0,2024-12-01,CUST-0001,network_egress,NaN,70.990000,NaN,74.00,0.15,NaN,4465.271000,NaN
1,2025-06-01,CUST-0001,network_egress,NaN,77.060000,NaN,74.00,0.15,NaN,4847.074000,NaN
2,2025-07-01,CUST-0001,gpu,NaN,3069.350000,NaN,2.65,0.15,NaN,6913.710875,NaN
3,2025-12-01,CUST-0001,network_egress,NaN,78.260000,NaN,74.00,0.15,NaN,4922.554000,NaN
4,2025-05-01,CUST-0002,object_storage,NaN,10.340000,NaN,22.00,0.05,NaN,216.106000,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8587,2026-07-01,CUST-0018,gpu,20880.72000,23020.993800,25265.67120,2.55,0.22,41531.752080,45788.756668,50253.420017
8588,2026-08-01,CUST-0018,gpu,20880.72000,24172.043490,27792.23832,2.55,0.22,41531.752080,48078.194502,55278.762018
8589,2026-06-01,CUST-0018,network_egress,923.22900,1033.183751,1128.39100,74.00,0.22,53288.777880,59635.366109,65130.728520
8590,2026-07-01,CUST-0018,network_egress,830.90610,1040.610506,1241.23010,74.00,0.22,47959.900092,60064.038419,71643.801372


In [5]:
df_historical_usage_aggrs = df_historical_usage.groupby(['pmonth']).agg(
    quantity_sum    = ('quantity', 'sum'),
    revenue_sum   = ('revenue', 'sum'),
    greatest_revenue_sum = ('greatest_revenue', 'sum'),
    least_revenue_sum = ('least_revenue', 'sum')
).reset_index()

In [6]:
fig = px.bar(
    df_historical_usage_aggrs,
    x="pmonth",
    y="quantity_sum",
    barmode="stack",
    title="Product usage per month",
    labels={"pmonth": "Month", "quantity_sum": "Product Metrics"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

In [7]:
fig = px.line(
    df_historical_usage_aggrs,
    x="pmonth",
    y=["revenue_sum", "greatest_revenue_sum", "least_revenue_sum"],
    title="Product revenue per month",
    labels={"pmonth": "Month", "revenue_sum": "Revenue"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

# Explore CRM opportunities

Clearly matched customers
- CUST-0038 — already included in the forecast (opportunity_id = OPP-1017); exclude from the downside
- CUST-0047 — already included in the forecast (opportunity_id = OPP-1021); exclude from the downside
- CUST-0016 — not included in the forecast
- CUST-0014 — not included in the forecast;


Deals and confidence
- opportunity_id = OPP-1036 — include in the base scenario
- opportunity_id = OPP-1037 — include in the upside scenario
- in the downside — do not include this deal at all
- opportunity_id = OPP-1039 — include everywhere

In [8]:
df_new_customers = (
    con.sql(
    """
    with cte_month as
        (SELECT *
         FROM (VALUES
                (DATE '2026-06-01'),
                (DATE '2026-07-01'),
                (DATE '2026-08-01')) AS t(pmonth))
    select pmonth,
           sum(if(opportunity_id in ('OPP-1039'), expected_monthly_consumption_usd, 0)) as least_revenue,
           sum(if(opportunity_id in ('OPP-1036', 'OPP-1039'), expected_monthly_consumption_usd, 0)) as revenue,
           sum(if(opportunity_id in ('OPP-1037', 'OPP-1039'), expected_monthly_consumption_usd, 0)) as greatest_revenue
    from crm_opportunities crm_opp
    left join cte_month cm
           on date(crm_opp.expected_start_month) <= date(cm.pmonth)
    where opportunity_id in ('OPP-1036', 'OPP-1037', 'OPP-1039')
    group by pmonth
    order by pmonth
    """).df())

In [9]:
df_new_customers

,pmonth,least_revenue,revenue,greatest_revenue
0,2026-06-01,150000.0,150000.0,150000.0
1,2026-07-01,150000.0,380000.0,840000.0
2,2026-08-01,150000.0,380000.0,840000.0


# Combine the forecast

In [10]:
df_historical_usage_aggrs = df_historical_usage.groupby(['pmonth']).agg(
    least_revenue = ('least_revenue', 'sum'),
    revenue   = ('revenue', 'sum'),
    greatest_revenue = ('greatest_revenue', 'sum'),
).reset_index()

In [11]:
df_historical_usage_aggrs

,pmonth,least_revenue,revenue,greatest_revenue
0,2024-01-01,0.000000e+00,4.718199e+05,0.000000e+00
1,2024-02-01,0.000000e+00,4.811140e+05,0.000000e+00
2,2024-03-01,0.000000e+00,4.988384e+05,0.000000e+00
3,2024-04-01,0.000000e+00,5.303208e+05,0.000000e+00
4,2024-05-01,0.000000e+00,5.460502e+05,0.000000e+00
5,2024-06-01,0.000000e+00,5.656783e+05,0.000000e+00
6,2024-07-01,0.000000e+00,6.342564e+05,0.000000e+00
7,2024-08-01,0.000000e+00,6.249028e+05,0.000000e+00
8,2024-09-01,0.000000e+00,6.797656e+05,0.000000e+00
9,2024-10-01,0.000000e+00,7.137797e+05,0.000000e+00


In [12]:
df = pd.DataFrame()
df = pd.concat([df_new_customers, df_historical_usage_aggrs], ignore_index=True)
df = df.groupby(['pmonth']).agg(
    least_revenue = ('least_revenue', 'sum'),
    revenue   = ('revenue', 'sum'),
    greatest_revenue = ('greatest_revenue', 'sum'),
).reset_index()

In [13]:
df

,pmonth,least_revenue,revenue,greatest_revenue
0,2024-01-01,0.000000e+00,4.718199e+05,0.000000e+00
1,2024-02-01,0.000000e+00,4.811140e+05,0.000000e+00
2,2024-03-01,0.000000e+00,4.988384e+05,0.000000e+00
3,2024-04-01,0.000000e+00,5.303208e+05,0.000000e+00
4,2024-05-01,0.000000e+00,5.460502e+05,0.000000e+00
5,2024-06-01,0.000000e+00,5.656783e+05,0.000000e+00
6,2024-07-01,0.000000e+00,6.342564e+05,0.000000e+00
7,2024-08-01,0.000000e+00,6.249028e+05,0.000000e+00
8,2024-09-01,0.000000e+00,6.797656e+05,0.000000e+00
9,2024-10-01,0.000000e+00,7.137797e+05,0.000000e+00


In [14]:
fig = px.line(
    df,
    x="pmonth",
    y=["revenue", "greatest_revenue", "least_revenue"],
    title="Product revenue per month"
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

In [15]:
df = df[
    (df["pmonth"] >= "2026-06-01") &
    (df["pmonth"] <= "2026-08-01")
]

In [16]:
df

,pmonth,least_revenue,revenue,greatest_revenue
29,2026-06-01,1.550957e+06,1.630922e+06,1.673158e+06
30,2026-07-01,1.528737e+06,1.898040e+06,2.446186e+06
31,2026-08-01,1.508562e+06,1.937226e+06,2.535340e+06


In [17]:
df_scenarios = (
    df.melt(
        id_vars=["pmonth"],
        value_vars=[
            "least_revenue",
            "revenue",
            "greatest_revenue",
        ],
        var_name="scenario",
        value_name="consumption_usd",
    )
    .replace({
        "scenario": {
            "least_revenue": "downside",
            "revenue": "base",
            "greatest_revenue": "upside",
        }
    })
)

df_scenarios["consumption_usd"] = (
    df_scenarios["consumption_usd"] / 1000
)

df_scenarios = df_scenarios.rename(
    columns={"consumption_usd": "consumption_usd_k"}
)

In [18]:
df_scenarios

,pmonth,scenario,consumption_usd_k
0,2026-06-01,downside,1550.957150
1,2026-07-01,downside,1528.737155
2,2026-08-01,downside,1508.561663
3,2026-06-01,base,1630.921500
4,2026-07-01,base,1898.040317
5,2026-08-01,base,1937.226188
6,2026-06-01,upside,1673.158283
7,2026-07-01,upside,2446.185527
8,2026-08-01,upside,2535.340417


# Cash — forecast by scenario

## Cash forecast logic

Cash is built from **three separate components** — the way money actually lands on the account:

> **Cash = A. Existing receivables + B. Forecast consumption × lag + C. New deals**

**A. Existing receivables (AR).** Take **real invoices** that, as of the forecast date (2026-06-06), are already issued but not yet paid (`invoice_date ≤ as-of` and not paid by that date):
- with a payment due date (`due_date + 7 days`) in June–August → collected in full that month (high likelihood);
- already overdue (due before June) → **recovery by debt age**: 1–30d 80%, 31–90d 50%, 91–180d 20%, 180+ 5% (≈999K older than 180 days — almost bad debt), spread over 3 months.

**B. Forecast consumption × lag.** Apply the payment lag from invoice history (**≈16% after 1 mo, ≈56% after 2 mo**) to the **June–August** forecast consumption (arrears customers only, excluding `annual_prepay`). Past months are **not** taken here — their money is already in receivables (A), otherwise it would be double counted. So most of the cash from new consumption "leaks" beyond August, and only the tail is visible in the window.

**C. New deals.** Annual prepayments in the start month = `committed_spend × 12 / term`:
- **OPP-1039** (Closed Won) → **1,800K in June, in all scenarios**
- **base:** + OPP-1036 → **2,300K** in July
- **upside:** + OPP-1037 → **6,900K** in July

Receivables (A) are identical across scenarios — it is already-existing debt; only B and C differ across scenarios.

In [19]:
# ── Full cash forecast in a single SQL query: A. receivables + B. forecast x lag + C. deals ──
# df_historical_usage (the consumption forecast above) is read directly in SQL.

df_cash_scenarios = con.sql(f"""
    with
    -- clean invoices: drop duplicates and test rows
    clean as (
        select * exclude(rn)
        from (select *,
                     row_number() over (partition by invoice_id order by invoice_date desc) as rn
            from invoices
            where invoice_type = 'usage'
            and payment_status <> 'void_pending')
        where rn = 1)

    -- PAYMENT LAG from history: how many months after consumption the money arrives.
    -- Only mature months (<= Feb-2026) and payments up to as-of -- no looking into the future.
    , lag_weights as (
        select datediff('month', service_month, date_trunc('month', paid_date)) as k,
               sum(amount_usd) / sum(sum(amount_usd)) over () as w
        from clean
        where payment_status = 'paid'
          and paid_date is not null
          and paid_date <= date '2026-06-06'
          and service_month is not null
          and service_month <= date '2026-02-01'
        group by 1)

    -- customer payment type (latest active contract; no contract = payg / arrears)
    , bill_freq as (
        select customer_id,
               billing_frequency
        from (
            select customer_id,
                   billing_frequency,
                   row_number() over (partition by customer_id order by contract_start desc) as rn
            from contracts
            where contract_status = 'active')
        where rn = 1
    )

    , forecast_months as (
        select *
        from (values (date '2026-06-01'),
                     (date '2026-07-01'),
                     (date '2026-08-01')) t(pmonth)
    ),

    -- ═══ A. EXISTING RECEIVABLES: invoices issued and unpaid as of as-of ═══
    open_ar as (
        select *, datediff('day', due_date, date '2026-06-06') as days_overdue
        from clean
        where invoice_date <= date '2026-06-06'
          and not (paid_date is not null and paid_date <= date '2026-06-06')
    ),
    -- near-term AR: due (due+7) in Jun-Aug -> collected in full that month
    ar_near as (
        select date_trunc('month', due_date + interval 7 day) as pmonth, sum(amount_usd) as ar
        from open_ar
        where due_date + interval 7 day between date '2026-06-01' and date '2026-08-31'
        group by 1
    ),
    -- overdue AR: recovery by debt age, spread over 3 months
    ar_overdue as (
        select sum(amount_usd * case when days_overdue <= 30  then 0.80
                                     when days_overdue <= 90  then 0.50
                                     when days_overdue <= 180 then 0.20
                                     else 0.05 end) / 3.0 as ar_per_month
        from open_ar where due_date + interval 7 day < date '2026-06-01'
    ),
    existing_ar as (
        select m.pmonth,
               coalesce(n.ar, 0) + (select ar_per_month from ar_overdue) as ar
        from forecast_months m left join ar_near n on n.pmonth = m.pmonth
    ),

    -- ═══ B. FORECAST CONSUMPTION x LAG: Jun-Aug only, arrears customers ═══
    arrears_fc as (
        select u.pmonth,
               sum(u.least_revenue) as downside, sum(u.revenue) as base, sum(u.greatest_revenue) as upside
        from df_historical_usage u
        left join bill_freq bf on u.customer_id = bf.customer_id
        where u.pmonth >= date '2026-06-01'
          and coalesce(bf.billing_frequency, 'payg') <> 'annual_prepay'
        group by 1
    ),
    future_usage as (
        select m.pmonth,
               sum(a.downside * lw.w) as downside, sum(a.base * lw.w) as base, sum(a.upside * lw.w) as upside
        from forecast_months m
        cross join lag_weights lw
        join arrears_fc a on datediff('month', a.pmonth, m.pmonth) = lw.k
        group by 1
    ),

    -- ═══ C. NEW DEALS: annual prepayments in the start month ═══
    prepay as (
        select date_trunc('month', expected_start_month) as pmonth,
               sum(case when opportunity_id = 'OPP-1039'               then expected_committed_spend_usd * 12.0 / contract_term_months else 0 end) as downside,
               sum(case when opportunity_id in ('OPP-1036','OPP-1039') then expected_committed_spend_usd * 12.0 / contract_term_months else 0 end) as base,
               sum(case when opportunity_id in ('OPP-1037','OPP-1039') then expected_committed_spend_usd * 12.0 / contract_term_months else 0 end) as upside
        from crm_opportunities where opportunity_id in ('OPP-1036', 'OPP-1037', 'OPP-1039')
        group by 1
    ),

    -- long format (month x scenario) for B and C; receivables A are the same across scenarios
    fu_long as (select pmonth, scenario, val as fu from future_usage unpivot (val for scenario in (downside, base, upside))),
    pp_long as (select pmonth, scenario, val as pp from prepay        unpivot (val for scenario in (downside, base, upside)))

    select f.pmonth,
           s.scenario,
           round(coalesce(ar.ar, 0) / 1000, 1)                                             as existing_ar_k,
           round(coalesce(fu.fu, 0) / 1000, 1)                                             as future_usage_k,
           round(coalesce(pp.pp, 0) / 1000, 1)                                             as prepay_k,
           round((coalesce(ar.ar, 0) + coalesce(fu.fu, 0) + coalesce(pp.pp, 0)) / 1000, 1) as cash_usd_k
    from forecast_months f
    cross join (values ('downside'), ('base'), ('upside')) s(scenario)
    left join existing_ar ar on ar.pmonth = f.pmonth
    left join fu_long fu on fu.pmonth = f.pmonth and fu.scenario = s.scenario
    left join pp_long pp on pp.pmonth = f.pmonth and pp.scenario = s.scenario
    order by s.scenario, f.pmonth
""").df()

In [20]:
df_cash_scenarios

,pmonth,scenario,existing_ar_k,future_usage_k,prepay_k,cash_usd_k
0,2026-06-01,base,1185.0,0.7,1800.0,2985.7
1,2026-07-01,base,1275.5,197.5,2300.0,3773.0
2,2026-08-01,base,371.2,882.3,0.0,1253.6
3,2026-06-01,downside,1185.0,0.6,1800.0,2985.6
4,2026-07-01,downside,1275.5,187.6,0.0,1463.1
5,2026-08-01,downside,371.2,830.2,0.0,1201.4
6,2026-06-01,upside,1185.0,0.7,1800.0,2985.7
7,2026-07-01,upside,1275.5,203.4,6900.0,8378.9
8,2026-08-01,upside,371.2,914.8,0.0,1286.0


In [21]:
fig = px.bar(
    df_cash_scenarios, x='pmonth', y='cash_usd_k', color='scenario', barmode='group',
    title='Cash collections forecast by scenario ($K)',
    labels={'pmonth': 'Month', 'cash_usd_k': 'Cash, $K', 'scenario': 'Scenario'}
)
fig.show()

In [22]:
# Summary: consumption and cash side by side (cash leads revenue in July due to the annual prepayment)
summary = (
    df_scenarios.assign(pmonth=pd.to_datetime(df_scenarios['pmonth']))
    .merge(df_cash_scenarios[['pmonth', 'scenario', 'cash_usd_k']], on=['pmonth', 'scenario'])
    .sort_values(['scenario', 'pmonth'])
    .reset_index(drop=True)
)

In [23]:
summary

,pmonth,scenario,consumption_usd_k,cash_usd_k
0,2026-06-01,base,1630.921500,2985.7
1,2026-07-01,base,1898.040317,3773.0
2,2026-08-01,base,1937.226188,1253.6
3,2026-06-01,downside,1550.957150,2985.6
4,2026-07-01,downside,1528.737155,1463.1
5,2026-08-01,downside,1508.561663,1201.4
6,2026-06-01,upside,1673.158283,2985.7
7,2026-07-01,upside,2446.185527,8378.9
8,2026-08-01,upside,2535.340417,1286.0


In [24]:
summary.to_csv('forecast_summary.csv', index=False)